In [ ]:
# AI-BASED REAL-TIME ANOMALY DETECTION & AUTONOMOUS DECISION


import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.animation import FuncAnimation
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from IPython.display import display, HTML, Image
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              confusion_matrix, mean_absolute_error, mean_squared_error)
from tensorflow import keras
from tensorflow.keras import layers, callbacks
warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#0a0f1e',
    'axes.facecolor':   '#0d1526',
    'axes.edgecolor':   '#1e3a5f',
    'axes.labelcolor':  '#7eb8d4',
    'xtick.color':      '#4a7a9b',
    'ytick.color':      '#4a7a9b',
    'text.color':       '#c8e0f4',
    'grid.color':       '#1e3a5f',
    'grid.alpha':       0.4,
})

display(HTML("""
<div style="background:linear-gradient(135deg,#0a0f1e,#0d2040);
            border:1px solid #1e3a5f;border-left:4px solid #00d4ff;
            border-radius:8px;padding:20px 28px;margin-bottom:16px;
            font-family:'Courier New',monospace;">
  <div style="color:#00d4ff;font-size:18px;font-weight:bold;letter-spacing:3px">
    MISSION CONTROL — DEEP SPACE ENGINE HEALTH MONITOR
  </div>
  <div style="color:#4a7a9b;font-size:11px;margin-top:6px;letter-spacing:2px">
    INDIA SPACE ACADEMY &nbsp;·&nbsp; NASA CMAPSS FD001
    &nbsp;·&nbsp; ISOLATION FOREST + AUTOENCODER + LSTM
  </div>
</div>
"""))

#%%
# ── 1: load data ───────────────────────────────────────────────────────────────
display(HTML('<div style="color:#00d4ff;font-family:monospace;letter-spacing:2px;margin:8px 0">[ 1/5 ] LOADING TELEMETRY DATA...</div>'))

PATH = '/kaggle/input/datasets/bishals098/nasa-turbofan-engine-degradation-simulation'
cols = ['unit','cycle','os1','os2','os3'] + [f's{i}' for i in range(1,22)]

train    = pd.read_csv(f'{PATH}/train_FD001.txt', sep=r'\s+', header=None, names=cols)
test     = pd.read_csv(f'{PATH}/test_FD001.txt',  sep=r'\s+', header=None, names=cols)
rul_true = pd.read_csv(f'{PATH}/RUL_FD001.txt',   header=None, names=['rul'])

train['max_cycle'] = train.groupby('unit')['cycle'].transform('max')
train['RUL']       = train['max_cycle'] - train['cycle']
train.drop(columns=['max_cycle'], inplace=True)

DROP  = ['unit','cycle','os1','os2','os3','s1','s5','s6','s10','s16','s18','s19']
SCOLS = [c for c in cols if c not in DROP]
N     = len(SCOLS)

scaler   = MinMaxScaler()
X_train  = scaler.fit_transform(train[SCOLS])
X_test   = scaler.transform(test[SCOLS])

DANGER   = 30
WINDOW   = 30
MAX_RUL  = 125
labels   = (train['RUL'] < DANGER).astype(int).values
X_health = X_train[labels == 0]
units    = train['unit'].values

print(f'train: {train.shape}  test: {test.shape}  features: {N}')
print(f'normal: {(labels==0).sum()}  degraded: {(labels==1).sum()}')

#%%
# ── 2: train models ────────────────────────────────────────────────────────────
display(HTML('<div style="color:#00d4ff;font-family:monospace;letter-spacing:2px;margin:8px 0">[ 2/5 ] TRAINING MODELS...</div>'))

# isolation forest
iso       = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
iso.fit(X_health)
iso_preds = (iso.predict(X_train) == -1).astype(int)   # shape: (n_train,)
print('isolation forest done')

# autoencoder
inp = keras.Input(shape=(N,))
x   = layers.Dense(10, activation='relu')(inp)
x   = layers.Dense(5,  activation='relu')(x)
x   = layers.Dense(10, activation='relu')(x)
out = layers.Dense(N,  activation='sigmoid')(x)
ae  = keras.Model(inp, out)
ae.compile(optimizer='adam', loss='mse')
ae.fit(X_health, X_health, epochs=30, batch_size=64, validation_split=0.1, verbose=0,
       callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
ae_err    = np.mean((X_train - ae.predict(X_train, verbose=0))**2, axis=1)
ae_thr    = np.percentile(np.mean((X_health - ae.predict(X_health, verbose=0))**2, axis=1), 95)
ae_preds  = (ae_err > ae_thr).astype(int)              # shape: (n_train,)
print('autoencoder done')

# ── lstm anomaly: build per-sample preds aligned to full train index ───────────
def make_windows(X, y, units_arr, w):
    Xl, yl, idx = [], [], []
    for uid in np.unique(units_arr):
        m  = np.where(units_arr == uid)[0]
        xs, ys = X[m], y[m]
        for i in range(len(xs)-w):
            Xl.append(xs[i:i+w]); yl.append(ys[i+w]); idx.append(m[i+w])
    return np.array(Xl), np.array(yl), np.array(idx)

Xw, yw, win_idx = make_windows(X_train, labels, units, WINDOW)

lstm_ad = keras.Sequential([
    layers.LSTM(64, return_sequences=True, input_shape=(WINDOW, N)),
    layers.Dropout(0.2),
    layers.LSTM(32),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1,  activation='sigmoid')
])
lstm_ad.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_ad.fit(Xw, yw, epochs=20, batch_size=256, validation_split=0.1, verbose=0,
            callbacks=[callbacks.EarlyStopping(patience=3, restore_best_weights=True)])

# map window predictions back to full training indices (pad early cycles with 0)
raw_probs   = lstm_ad.predict(Xw, verbose=0).flatten()
lstm_preds  = np.zeros(len(train), dtype=int)          # shape: (n_train,) — fixed!
lstm_preds[win_idx] = (raw_probs >= 0.5).astype(int)
print('lstm anomaly classifier done')

# ── lstm rul ───────────────────────────────────────────────────────────────────
train['RUL_cap'] = train['RUL'].clip(upper=MAX_RUL)

def make_windows_rul(X, rul, units_arr, w):
    Xl, yl = [], []
    for uid in np.unique(units_arr):
        m  = units_arr == uid
        xs, rs = X[m], rul[m]
        for i in range(len(xs)-w):
            Xl.append(xs[i:i+w]); yl.append(rs[i+w])
    return np.array(Xl), np.array(yl)

Xr, yr   = make_windows_rul(X_train, train['RUL_cap'].values, units, WINDOW)
lstm_rul = keras.Sequential([
    layers.LSTM(64, return_sequences=True, input_shape=(WINDOW, N)),
    layers.Dropout(0.2),
    layers.LSTM(32),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(1,  activation='sigmoid')
])
lstm_rul.compile(optimizer='adam', loss='mse', metrics=['mae'])
lstm_rul.fit(Xr, yr/MAX_RUL, epochs=30, batch_size=256, validation_split=0.1, verbose=0,
             callbacks=[
                 callbacks.EarlyStopping(patience=5, restore_best_weights=True),
                 callbacks.ReduceLROnPlateau(patience=3, factor=0.5, min_lr=1e-6)
             ])
print('lstm rul regressor done')

# test set predictions
def get_last_window(X, units_arr, w):
    Xl = []
    for uid in sorted(np.unique(units_arr)):
        m  = units_arr == uid; xs = X[m]
        if len(xs) < w:
            xs = np.vstack([np.tile(xs[0], (w-len(xs), 1)), xs])
        Xl.append(xs[-w:])
    return np.array(Xl)

X_test_win = get_last_window(X_test, test['unit'].values, WINDOW)
rul_pred   = lstm_rul.predict(X_test_win, verbose=0).flatten() * MAX_RUL
rul_actual = rul_true['rul'].values
mae        = mean_absolute_error(rul_actual, rul_pred)
rmse       = np.sqrt(mean_squared_error(rul_actual, rul_pred))

#%%
# ── 3: results + static plots ─────────────────────────────────────────────────
display(HTML('<div style="color:#00d4ff;font-family:monospace;letter-spacing:2px;margin:8px 0">[ 3/5 ] RESULTS AND PLOTS</div>'))

# all three preds are now shape (n_train,) — no index mismatch possible
print('===== ANOMALY DETECTION RESULTS =====')
print(f'{"model":<22} {"precision":>10} {"recall":>10} {"f1":>8}')
print('-' * 52)
for name, preds in [('isolation forest', iso_preds),
                    ('autoencoder',      ae_preds),
                    ('lstm classifier',  lstm_preds)]:
    print(f'{name:<22}'
          f' {precision_score(labels,preds,zero_division=0):>10.3f}'
          f' {recall_score(labels,preds,zero_division=0):>10.3f}'
          f' {f1_score(labels,preds,zero_division=0):>8.3f}')
print(f'\nRUL:  MAE={mae:.2f}  RMSE={rmse:.2f}')
c_e = sum(r<30 for r in rul_pred)
w_e = sum(30<=r<80 for r in rul_pred)
n_e = sum(r>=80 for r in rul_pred)
print(f'fleet: normal={n_e}  warning={w_e}  critical={c_e}')

# confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle('CONFUSION MATRICES', color='#00d4ff', fontsize=12, fontfamily='monospace')
for ax, (name, preds) in zip(axes, [
    ('Isolation Forest', iso_preds),
    ('Autoencoder',      ae_preds),
    ('LSTM Classifier',  lstm_preds)
]):
    sns.heatmap(confusion_matrix(labels, preds), annot=True, fmt='d', cmap='Blues',
                ax=ax, xticklabels=['normal','degraded'], yticklabels=['normal','degraded'],
                linewidths=0.5, linecolor='#1e3a5f')
    ax.set_title(name, color='#00d4ff', fontsize=10, fontfamily='monospace')
    ax.set_xlabel('predicted', fontsize=8); ax.set_ylabel('actual', fontsize=8)
plt.tight_layout()
plt.savefig('plot_confusion_matrices.png', dpi=120, bbox_inches='tight', facecolor='#0a0f1e')
plt.show()

# rul scatter + error
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('RUL PREDICTION RESULTS', color='#00d4ff', fontsize=12, fontfamily='monospace')
axes[0].scatter(rul_actual, rul_pred, alpha=0.5, color='#00d4ff', s=30)
mv = max(rul_actual.max(), rul_pred.max())
axes[0].plot([0,mv],[0,mv], color='#ff2b4a', ls='--', lw=2, label='perfect')
axes[0].set(title=f'predicted vs actual  MAE={mae:.1f}', xlabel='true RUL', ylabel='predicted RUL')
axes[0].legend(); axes[0].grid(True)
axes[1].hist(rul_pred-rul_actual, bins=30, color='#00d4ff', edgecolor='#0a0f1e', alpha=0.8)
axes[1].axvline(0, color='#ff2b4a', ls='--', lw=2)
axes[1].set(title='error distribution', xlabel='error (cycles)', ylabel='count')
axes[1].grid(True)
plt.tight_layout()
plt.savefig('plot_rul_results.png', dpi=120, bbox_inches='tight', facecolor='#0a0f1e')
plt.show()

# fleet bar
bar_colors = ['#ff2b4a' if r<30 else '#ff8c00' if r<80 else '#00ff88' for r in rul_pred]
fig, ax = plt.subplots(figsize=(14, 4))
fig.suptitle('FLEET RUL — 100 TEST ENGINES', color='#00d4ff', fontsize=12, fontfamily='monospace')
ax.bar(range(1,101), rul_pred, color=bar_colors, alpha=0.85)
ax.axhline(30, color='#ff2b4a', ls='--', lw=1.5, label='critical < 30')
ax.axhline(80, color='#ff8c00', ls='--', lw=1.5, label='warning < 80')
ax.set_xlabel('engine id'); ax.set_ylabel('predicted RUL')
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig('plot_fleet.png', dpi=120, bbox_inches='tight', facecolor='#0a0f1e')
plt.show()

#%%
# ── 4: autonomous decision engine ─────────────────────────────────────────────
display(HTML('<div style="color:#00d4ff;font-family:monospace;letter-spacing:2px;margin:8px 0">[ 4/5 ] AUTONOMOUS DECISION ENGINE</div>'))

def risk_score(iv, av, pred_rul):
    return round(0.4*((iv+av)/2) + 0.6*(1 - min(pred_rul/MAX_RUL, 1.0)), 3)

def get_action(risk):
    if   risk < 0.25: return 'NORMAL',    'continue ops',   '#00ff88'
    elif risk < 0.45: return 'MONITOR',   'send telemetry', '#00d4ff'
    elif risk < 0.65: return 'WARNING',   'reduce load',    '#ff8c00'
    elif risk < 0.80: return 'HIGH RISK', 'switch backup',  '#ff6600'
    else:             return 'EMERGENCY', 'shutdown now',   '#ff2b4a'

action_counts = {'NORMAL':0,'MONITOR':0,'WARNING':0,'HIGH RISK':0,'EMERGENCY':0}
for i in range(len(rul_pred)):
    rows = np.where(units == (i+1))[0]
    iv   = iso_preds[rows[-1]] if len(rows) else 0
    av   = ae_preds[rows[-1]]  if len(rows) else 0
    r    = risk_score(iv, av, rul_pred[i])
    st, _, _ = get_action(r)
    action_counts[st] += 1
print(f'action distribution: {action_counts}')

#%%
# ── 5: real-time animation ─────────────────────────────────────────────────────
display(HTML('<div style="color:#00d4ff;font-family:monospace;letter-spacing:2px;margin:8px 0">[ 5/5 ] REAL-TIME ANOMALY DETECTION FEED — RENDERING...</div>'))

WATCH_UNIT   = 3       # engine with clear degradation
WATCH_SENSOR = 's2'
TRAIL        = 60

eng_mask  = units == WATCH_UNIT
eng_data  = train[eng_mask].reset_index(drop=True)
eng_X     = X_train[eng_mask]
eng_rul   = train['RUL'].values[eng_mask]
eng_iso   = iso_preds[eng_mask]    # already full-length, slice is safe
eng_ae    = ae_preds[eng_mask]
eng_lstm  = lstm_preds[eng_mask]   # also full-length now
total_cyc = len(eng_data)
danger_cyc= eng_data['cycle'].max() - DANGER

# composite risk per cycle using actual model flags
risks = np.array([
    risk_score(int(eng_iso[i]), int(eng_ae[i]), eng_rul[i])
    for i in range(total_cyc)
])

alerts = []

fig = plt.figure(figsize=(14, 9), facecolor='#0a0f1e')
fig.suptitle(f'REAL-TIME ENGINE {WATCH_UNIT} MONITOR — LIVE FEED',
             color='#00d4ff', fontsize=13, fontfamily='monospace', fontweight='bold', y=0.98)

gs       = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.35)
ax_s     = fig.add_subplot(gs[0, :])
ax_r     = fig.add_subplot(gs[1, :2])
ax_rul_g = fig.add_subplot(gs[1, 2])
ax_a     = fig.add_subplot(gs[2, :])

for ax in [ax_s, ax_r, ax_rul_g, ax_a]:
    ax.set_facecolor('#0d1526')
    for sp in ax.spines.values(): sp.set_edgecolor('#1e3a5f')

def frame_color(rul_v, risk_v):
    if rul_v < DANGER or risk_v >= 0.65: return '#ff2b4a'
    if rul_v < 80    or risk_v >= 0.45:  return '#ff8c00'
    return '#00ff88'

def animate(frame):
    i        = min(frame + 1, total_cyc - 1)
    start    = max(0, i - TRAIL)
    cyc_win  = eng_data['cycle'].values[start:i+1]
    sens_win = eng_data[WATCH_SENSOR].values[start:i+1]
    any_anom = (eng_iso[start:i+1]==1) | (eng_ae[start:i+1]==1) | (eng_lstm[start:i+1]==1)
    cur_rul  = eng_rul[i]
    cur_risk = risks[i]
    cur_cyc  = eng_data['cycle'].values[i]
    color    = frame_color(cur_rul, cur_risk)
    st, ac, _= get_action(cur_risk)

    # sensor stream
    ax_s.cla(); ax_s.set_facecolor('#0d1526')
    for sp in ax_s.spines.values(): sp.set_edgecolor('#1e3a5f')
    ax_s.fill_between(cyc_win, sens_win, alpha=0.1, color=color)
    ax_s.plot(cyc_win, sens_win, color=color, lw=1.3)
    ax_s.axvline(danger_cyc, color='#ff2b4a', ls=':', lw=1, alpha=0.5, label='danger zone')
    if any_anom.any():
        ax_s.scatter(cyc_win[any_anom], sens_win[any_anom],
                     color='#ff2b4a', s=28, zorder=6, label='anomaly detected')
    ax_s.set_xlim(cyc_win[0], cyc_win[0]+TRAIL+2)
    ax_s.set_ylabel(WATCH_SENSOR, fontsize=8)
    ax_s.set_title(
        f'cycle {cur_cyc:3d}  |  RUL {cur_rul:4.0f} cyc  |  risk {cur_risk:.3f}  |  {st}',
        color=color, fontsize=9, fontfamily='monospace', pad=3)
    ax_s.legend(fontsize=7, loc='upper left',
                facecolor='#0d1526', edgecolor='#1e3a5f', labelcolor='#c8e0f4')
    ax_s.grid(True, alpha=0.3)

    # risk timeline
    ax_r.cla(); ax_r.set_facecolor('#0d1526')
    for sp in ax_r.spines.values(): sp.set_edgecolor('#1e3a5f')
    risk_win = risks[start:i+1]
    for j in range(len(cyc_win)-1):
        rc = frame_color(eng_rul[start+j], risk_win[j])
        ax_r.plot(cyc_win[j:j+2], risk_win[j:j+2], color=rc, lw=1.5)
    for thr, col, lbl in [(0.65,'#ff2b4a','high risk'),(0.45,'#ff8c00','warning'),(0.25,'#00ff88','monitor')]:
        ax_r.axhline(thr, color=col, ls='--', lw=0.8, alpha=0.6, label=lbl)
    ax_r.set_ylim(0, 1); ax_r.set_xlim(cyc_win[0], cyc_win[0]+TRAIL+2)
    ax_r.set_ylabel('risk score', fontsize=8); ax_r.set_xlabel('cycle', fontsize=8)
    ax_r.set_title('composite risk score', color='#00d4ff', fontsize=9, fontfamily='monospace')
    ax_r.legend(fontsize=7, loc='upper left', facecolor='#0d1526',
                edgecolor='#1e3a5f', labelcolor='#c8e0f4')
    ax_r.grid(True, alpha=0.3)

    # RUL gauge
    ax_rul_g.cla(); ax_rul_g.axis('off')
    ax_rul_g.set_xlim(0,1); ax_rul_g.set_ylim(0,1)
    pct = min(cur_rul/MAX_RUL, 1.0)
    ax_rul_g.add_patch(FancyBboxPatch((0.08,0.18),0.84,0.10,
        boxstyle='round,pad=0.01', facecolor='#1e3a5f', edgecolor='none'))
    ax_rul_g.add_patch(FancyBboxPatch((0.08,0.18),0.84*pct,0.10,
        boxstyle='round,pad=0.01', facecolor=color, edgecolor='none', alpha=0.85))
    ax_rul_g.text(0.5,0.72,f'{cur_rul:.0f}', ha='center', va='center',
                  color=color, fontsize=30, fontfamily='monospace', fontweight='bold')
    ax_rul_g.text(0.5,0.52,'cycles remaining', ha='center', va='center',
                  color='#4a7a9b', fontsize=8, fontfamily='monospace')
    ax_rul_g.text(0.5,0.06, ac.upper(), ha='center', va='center',
                  color=color, fontsize=9, fontfamily='monospace', fontweight='bold')
    ax_rul_g.set_title('RUL', color='#00d4ff', fontsize=9, fontfamily='monospace')

    # alert log
    if any_anom[-1]:
        alerts.append((cur_cyc, st, ac, color))
    ax_a.cla(); ax_a.axis('off'); ax_a.set_facecolor('#0d1526')
    ax_a.set_title('ANOMALY ALERT LOG', color='#00d4ff',
                   fontsize=9, fontfamily='monospace', loc='left', pad=4)
    for k, (cyc, s, a, col) in enumerate(reversed(alerts[-6:])):
        ax_a.text(0.01, 0.85-k*0.16,
                  f'T+{cyc:03d}  ENG-{WATCH_UNIT:02d}  [{s}]  {a.upper()}',
                  transform=ax_a.transAxes, color=col,
                  fontsize=9, fontfamily='monospace', alpha=max(0.3, 1.0-k*0.15))
    if not alerts:
        ax_a.text(0.01, 0.5, 'ALL SYSTEMS NOMINAL',
                  transform=ax_a.transAxes, color='#00ff88',
                  fontsize=9, fontfamily='monospace')

ani = FuncAnimation(fig, animate, frames=min(total_cyc-1, 180),
                    interval=120, repeat=False, blit=False)
plt.tight_layout(rect=[0, 0, 1, 0.96])
ani.save('realtime_monitor.gif', writer='pillow', fps=8, dpi=90)
plt.close()

display(Image('realtime_monitor.gif'))
print('\nall outputs saved:')
print('  plot_confusion_matrices.png')
print('  plot_rul_results.png')
print('  plot_fleet.png')
print('  realtime_monitor.gif')

from IPython.display import FileLink, display

print("\n📥 CLICK TO DOWNLOAD:")
display(FileLink('realtime_monitor.gif'))
display(FileLink('plot_confusion_matrices.png'))
display(FileLink('plot_rul_results.png'))
display(FileLink('plot_fleet.png'))